# 14 — Final Interpretive Analysis and Discussion

**Project:** *Online Attention, Network Centrality, and Electoral Outcomes: A Network Science Study of Philippine Election 2025 Twitter/X Discourse*

This notebook turns the computed outputs from Notebooks 01–13 into a polished research interpretation. It is intentionally **visual-first**: each major claim is supported by a chart, network graph, heatmap, or diagnostic visualization.

## What this notebook adds

1. Executive research findings
2. Online-vs-election outcome alignment
3. Network science interpretation
4. Hashtag and candidate network interpretation
5. Regional/NCR comparison
6. Overrepresentation and underrepresentation analysis
7. Final conclusion, limitations, and report-ready discussion

> Run this notebook **after** Notebooks 01–13. If some files are missing, the notebook will show which upstream notebook must be rerun.


## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("14_final_interpretive_analysis_and_discussion", total_steps=22, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## 0. Setup

This cell loads the project paths, plotting libraries, and helper functions. It also sets a consistent visual style for the final report outputs.

In [ ]:
progress.step("0. Setup")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)
RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

print("Project root:", ROOT)
print("Processed data:", PROCESSED)
print("Figures:", FIGURES)
print("Tables:", TABLES)

## 1. Load final output tables

The notebook reads the key result tables generated by the previous notebooks. The goal here is not to recompute everything from scratch, but to interpret the final results consistently.

In [ ]:
progress.step("1. Load final output tables")
def load_first_available(name, candidates, required=False):
    # Load the first existing CSV path from a list of candidate paths.
    for p in candidates:
        p = Path(p)
        if p.exists():
            df = pd.read_csv(p)
            try:
                shown_path = p.relative_to(ROOT)
            except Exception:
                shown_path = p
            print(f"Loaded {name}: {shown_path} | shape={df.shape}")
            return df
    msg = f"Missing {name}. Checked: " + "; ".join(str(p) for p in candidates)
    if required:
        raise FileNotFoundError(msg)
    print("⚠️", msg)
    return pd.DataFrame()

national = load_first_available("national Senate results", [
    PROCESSED / "senate_national_candidate_results.csv",
    PROCESSED / "senate_national.csv",
])

comparison = load_first_available("candidate online vs election comparison", [
    PROCESSED / "candidate_online_vs_election_metrics.csv",
    PROCESSED / "comparison.csv",
])

correlations = load_first_available("online metric correlations", [
    TABLES / "10_correlation_online_metrics_vs_votes.csv",
    PROCESSED / "correlations.csv",
])

top12_overlap = load_first_available("top-12 overlap", [
    TABLES / "10_top12_overlap_precision_recall.csv",
    PROCESSED / "top12_overlap.csv",
])

hashtag_metrics = load_first_available("hashtag node metrics", [
    PROCESSED / "hashtag_node_metrics.csv",
    PROCESSED / "hashtag_metrics.csv",
])

candidate_metrics = load_first_available("candidate co-mention node metrics", [
    PROCESSED / "candidate_comention_node_metrics.csv",
    PROCESSED / "candidate_comention_metrics.csv",
])

candidate_edges = load_first_available("candidate co-mention edges", [
    PROCESSED / "candidate_comention_edges.csv",
])

candidate_hashtag_diversity = load_first_available("candidate hashtag diversity", [
    PROCESSED / "candidate_hashtag_diversity.csv",
])

candidate_hashtag_metrics = load_first_available("candidate-hashtag bipartite metrics", [
    PROCESSED / "candidate_hashtag_bipartite_node_metrics.csv",
    PROCESSED / "candidate_hashtag_metrics.csv",
])

regional_turnout = load_first_available("regional turnout", [
    PROCESSED / "region_turnout_metrics.csv",
    PROCESSED / "regional_turnout.csv",
])

ncr_vs_national = load_first_available("NCR vs national ranks", [
    TABLES / "11_ncr_vs_national_candidate_ranks.csv",
    PROCESSED / "ncr_vs_national.csv",
])

hashtag_edges = load_first_available("hashtag edges", [
    PROCESSED / "hashtag_edges_filtered_weight2.csv",
    PROCESSED / "hashtag_edges_filtered.csv",
    PROCESSED / "hashtag_edges_full.csv",
])

### NCR-vs-national self-check

This cell validates the NCR rank table. If the table is missing or incomplete, it rebuilds it directly from the raw Senate results file.

In [ ]:
progress.step("NCR-vs-national self-check")

def validate_ncr_vs_national_table(df):
    required_cols = {"candidate", "regional_rank", "vote_rank", "votes", "total_votes", "winner_top12", "ncr_minus_national_rank"}
    return isinstance(df, pd.DataFrame) and (not df.empty) and required_cols.issubset(df.columns) and len(df) >= 12


def rebuild_ncr_vs_national_from_raw():
    """Rebuild the NCR-vs-national rank table directly from the raw Senate file.

    This makes Notebook 14 self-healing: even if Notebook 11 was not run, or if
    the intermediate NCR table is missing/incomplete, this cell can recreate it
    from `data/raw/senate25-final_updated.csv`.
    """
    raw_senate_path = RAW / "senate25-final_updated.csv"
    if not raw_senate_path.exists():
        print("⚠️ Cannot rebuild NCR table because raw Senate file is missing:", raw_senate_path)
        return pd.DataFrame()

    senate_raw = pd.read_csv(raw_senate_path, low_memory=False)
    candidate_cols = get_candidate_columns(senate_raw)
    if not candidate_cols or "region" not in senate_raw.columns:
        print("⚠️ Cannot rebuild NCR table because candidate columns or region column are missing.")
        return pd.DataFrame()

    # Build or rebuild national result table when the loaded one is unavailable.
    if validate_ncr_vs_national_table(ncr_vs_national) and not national.empty:
        national_for_merge = national.copy()
    elif not national.empty and {"candidate_column", "vote_rank", "total_votes", "winner_top12"}.issubset(national.columns):
        national_for_merge = national.copy()
    else:
        national_for_merge = senate_national_results(senate_raw)
        national_for_merge.to_csv(PROCESSED / "senate_national_candidate_results.csv", index=False)

    ncr_mask = senate_raw["region"].astype(str).str.upper().str.contains("NATIONAL CAPITAL REGION|\\bNCR\\b", regex=True, na=False)
    if ncr_mask.sum() == 0:
        print("⚠️ No NCR rows found. Available region values:")
        display(pd.DataFrame({"region": sorted(senate_raw["region"].dropna().astype(str).unique())}))
        return pd.DataFrame()

    ncr_raw = senate_raw.loc[ncr_mask].copy()
    ncr_votes = ncr_raw[candidate_cols].apply(pd.to_numeric, errors="coerce").sum().reset_index()
    ncr_votes.columns = ["candidate_column", "votes"]

    ref = build_candidate_reference(senate_raw)
    out = ref.merge(ncr_votes, on="candidate_column", how="left")
    out["region"] = "NATIONAL CAPITAL REGION"
    out["region_all_candidate_votes"] = out["votes"].sum()
    out["regional_vote_share"] = np.where(out["region_all_candidate_votes"] > 0, out["votes"] / out["region_all_candidate_votes"], np.nan)
    out["regional_rank"] = out["votes"].rank(method="min", ascending=False).astype(int)

    keep_nat = [c for c in ["candidate_column", "vote_rank", "total_votes", "winner_top12"] if c in national_for_merge.columns]
    out = out.merge(national_for_merge[keep_nat], on="candidate_column", how="left")
    if "winner_top12" not in out.columns and "vote_rank" in out.columns:
        out["winner_top12"] = out["vote_rank"] <= 12
    out["ncr_minus_national_rank"] = out["regional_rank"] - out["vote_rank"]

    final_cols = [
        "region", "candidate_column", "ballot_no", "candidate", "party", "last_name", "first_name",
        "votes", "region_all_candidate_votes", "regional_vote_share", "regional_rank",
        "vote_rank", "total_votes", "winner_top12", "ncr_minus_national_rank"
    ]
    out = out[[c for c in final_cols if c in out.columns]].sort_values("regional_rank").reset_index(drop=True)

    out.to_csv(PROCESSED / "ncr_vs_national.csv", index=False)
    out.to_csv(TABLES / "11_ncr_vs_national_candidate_ranks.csv", index=False)
    out.to_csv(TABLES / "14_ncr_vs_national_full_table.csv", index=False)
    print(f"Rebuilt NCR vs national rank table: {out.shape[0]} candidates")
    return out


if not validate_ncr_vs_national_table(ncr_vs_national):
    print("NCR vs national table is missing or incomplete. Rebuilding from raw Senate results...")
    ncr_vs_national = rebuild_ncr_vs_national_from_raw()
else:
    print(f"NCR vs national table validated: {ncr_vs_national.shape[0]} candidates")
    ncr_vs_national.to_csv(TABLES / "14_ncr_vs_national_full_table.csv", index=False)


### Comparison-table self-repair

Some earlier project versions produced the online-vs-election comparison table without rank and overperformance columns. This cell standardizes the table so later graphs do not disappear when those columns are missing.


In [ ]:
progress.step("Comparison-table self-repair")
def standardize_comparison_table(comparison, national=None, save_outputs=True):
    """Ensure the comparison table has the rank and overperformance fields used by Notebook 14.

    Why this exists:
    - Earlier Notebook 10 versions exported only base online metrics.
    - Notebook 14 needs rank and overperformance columns for scatterplots and bar charts.
    - This function repairs the table in place when those columns are missing.
    """
    if comparison is None or comparison.empty:
        print("⚠️ Cannot standardize comparison table because it is empty.")
        return pd.DataFrame()

    df = comparison.copy()

    # Attach national election fields if needed and if a national table is available.
    if national is not None and not national.empty and "candidate" in df.columns and "candidate" in national.columns:
        needed_from_national = [c for c in ["candidate", "total_votes", "vote_rank", "winner_top12", "party"] if c in national.columns]
        for col in ["total_votes", "vote_rank", "winner_top12", "party"]:
            if col not in df.columns and col in needed_from_national:
                df = df.merge(national[needed_from_national].drop_duplicates("candidate"), on="candidate", how="left")
                break

    if "vote_rank" in df.columns:
        df["vote_rank"] = pd.to_numeric(df["vote_rank"], errors="coerce")
    elif "total_votes" in df.columns:
        df["total_votes"] = pd.to_numeric(df["total_votes"], errors="coerce")
        df["vote_rank"] = df["total_votes"].rank(method="min", ascending=False)

    if "winner_top12" not in df.columns and "vote_rank" in df.columns:
        df["winner_top12"] = df["vote_rank"] <= 12

    candidate_online_metrics = [
        "twitter_mention_count",
        "total_engagement",
        "viewCount",
        "retweetCount",
        "likeCount",
        "quoteCount",
        "comention_pagerank",
        "comention_weighted_degree",
        "comention_betweenness_centrality",
        "bipartite_pagerank",
        "bipartite_weighted_degree",
        "bipartite_betweenness_centrality",
        "unique_hashtags",
        "total_candidate_hashtag_weight",
    ]

    created_rank_cols = []
    created_overperf_cols = []
    for metric in candidate_online_metrics:
        if metric not in df.columns:
            continue
        df[metric] = pd.to_numeric(df[metric], errors="coerce").fillna(0)
        rank_col = f"{metric}_rank"
        over_col = f"{metric}_overperformance"
        if rank_col not in df.columns:
            df[rank_col] = df[metric].rank(method="min", ascending=False)
            created_rank_cols.append(rank_col)
        else:
            df[rank_col] = pd.to_numeric(df[rank_col], errors="coerce")
        if "vote_rank" in df.columns and over_col not in df.columns:
            # Positive = stronger online rank than actual vote rank.
            df[over_col] = df["vote_rank"] - df[rank_col]
            created_overperf_cols.append(over_col)

    # Rebuild correlations if correlation table is unavailable or stale.
    if "total_votes" in df.columns:
        corr_rows = []
        for metric in candidate_online_metrics:
            if metric in df.columns and df[metric].notna().sum() >= 3:
                sub = df[[metric, "total_votes"]].dropna()
                if len(sub) >= 3:
                    try:
                        spearman_r, spearman_p = spearmanr(sub[metric], sub["total_votes"])
                        pearson_r, pearson_p = pearsonr(sub[metric], sub["total_votes"])
                        kendall_r, kendall_p = kendalltau(sub[metric], sub["total_votes"])
                        corr_rows.append({
                            "online_metric": metric,
                            "spearman_r_vs_votes": spearman_r,
                            "spearman_p_value": spearman_p,
                            "pearson_r_vs_votes": pearson_r,
                            "pearson_p_value": pearson_p,
                            "kendall_tau_vs_votes": kendall_r,
                            "kendall_p_value": kendall_p,
                            "n_candidates": len(sub),
                        })
                    except Exception:
                        pass
        if corr_rows:
            repaired_corr = pd.DataFrame(corr_rows).sort_values("spearman_r_vs_votes", ascending=False)
            repaired_corr.to_csv(PROCESSED / "correlations.csv", index=False)
            repaired_corr.to_csv(TABLES / "14_repaired_correlation_online_metrics_vs_votes.csv", index=False)
            globals()["correlations"] = repaired_corr

    # Rebuild Top-12 overlap table.
    if "winner_top12" in df.columns:
        overlap_rows = []
        actual_winners = set(df.loc[df["winner_top12"].astype(bool), "candidate"])
        for metric in candidate_online_metrics:
            rank_col = f"{metric}_rank"
            if rank_col in df.columns:
                online_top12 = set(df.sort_values(rank_col).head(12)["candidate"])
                overlap = len(actual_winners.intersection(online_top12))
                overlap_rows.append({
                    "online_metric": metric,
                    "top12_overlap_count": overlap,
                    "precision_at_12": overlap / 12,
                    "recall_at_12": overlap / 12,
                    "online_top12_candidates": ", ".join(df.sort_values(rank_col).head(12)["candidate"].astype(str).tolist()),
                })
        if overlap_rows:
            repaired_overlap = pd.DataFrame(overlap_rows).sort_values("top12_overlap_count", ascending=False)
            repaired_overlap.to_csv(PROCESSED / "top12_overlap.csv", index=False)
            repaired_overlap.to_csv(TABLES / "14_repaired_top12_overlap.csv", index=False)
            globals()["top12_overlap"] = repaired_overlap

    if save_outputs:
        df.to_csv(PROCESSED / "candidate_online_vs_election_metrics.csv", index=False)
        df.to_csv(PROCESSED / "comparison.csv", index=False)
        df.to_csv(TABLES / "14_repaired_candidate_online_vs_election_metrics.csv", index=False)

    print(f"Comparison table standardized: {df.shape[0]} candidates, {df.shape[1]} columns")
    if created_rank_cols:
        print(f"Created {len(created_rank_cols)} rank columns.")
    if created_overperf_cols:
        print(f"Created {len(created_overperf_cols)} overperformance columns.")
    return df

# scipy stats are used here only if correlations need repair.
try:
    from scipy.stats import spearmanr, pearsonr, kendalltau
except Exception as e:
    print("⚠️ scipy is not available; rank/overperformance repair will still work, but correlation repair will be skipped.", e)
    def spearmanr(*args, **kwargs): return (np.nan, np.nan)
    def pearsonr(*args, **kwargs): return (np.nan, np.nan)
    def kendalltau(*args, **kwargs): return (np.nan, np.nan)

comparison = standardize_comparison_table(comparison, national)


## 2. Readiness check

This section confirms whether the needed tables exist. Missing files usually mean the relevant upstream notebook has not yet been run.

In [ ]:
progress.step("2. Readiness check")
readiness = pd.DataFrame([
    {"Result table": "National Senate results", "Rows": len(national), "Status": "Ready" if not national.empty else "Missing"},
    {"Result table": "Online vs election comparison", "Rows": len(comparison), "Status": "Ready" if not comparison.empty else "Missing"},
    {"Result table": "Correlations", "Rows": len(correlations), "Status": "Ready" if not correlations.empty else "Missing"},
    {"Result table": "Top-12 overlap", "Rows": len(top12_overlap), "Status": "Ready" if not top12_overlap.empty else "Missing"},
    {"Result table": "Hashtag network metrics", "Rows": len(hashtag_metrics), "Status": "Ready" if not hashtag_metrics.empty else "Missing"},
    {"Result table": "Candidate co-mention metrics", "Rows": len(candidate_metrics), "Status": "Ready" if not candidate_metrics.empty else "Missing"},
    {"Result table": "Candidate-hashtag diversity", "Rows": len(candidate_hashtag_diversity), "Status": "Ready" if not candidate_hashtag_diversity.empty else "Missing"},
    {"Result table": "Regional turnout / geography", "Rows": len(regional_turnout), "Status": "Ready" if not regional_turnout.empty else "Missing"},
])
display(readiness)

missing = readiness.loc[readiness["Status"].eq("Missing"), "Result table"].tolist()
if missing:
    display(Markdown("### ⚠️ Missing upstream outputs\n" + "\n".join(f"- {m}" for m in missing)))
else:
    display(Markdown("### ✅ All major final-result tables are available."))

## 3. Executive summary cards

These cards summarize the central result of the project: the online network reflects electoral prominence, but imperfectly.

In [ ]:
progress.step("3. Executive summary cards")
# Derive key summary values safely.
n_candidates = int(national["candidate"].nunique()) if "candidate" in national.columns and not national.empty else "—"
winners = int(national["winner_top12"].sum()) if "winner_top12" in national.columns and not national.empty else 12

best_corr_metric = "—"
best_spearman = np.nan
if not correlations.empty and "spearman_r_vs_votes" in correlations.columns:
    tmp = correlations.dropna(subset=["spearman_r_vs_votes"]).sort_values("spearman_r_vs_votes", ascending=False)
    if len(tmp):
        best_corr_metric = tmp.iloc[0]["online_metric"]
        best_spearman = tmp.iloc[0]["spearman_r_vs_votes"]

best_overlap = np.nan
best_overlap_metric = "—"
if not top12_overlap.empty and "top12_overlap_count" in top12_overlap.columns:
    tmp = top12_overlap.dropna(subset=["top12_overlap_count"]).sort_values("top12_overlap_count", ascending=False)
    if len(tmp):
        best_overlap = tmp.iloc[0]["top12_overlap_count"]
        best_overlap_metric = tmp.iloc[0]["online_metric"]

n_hashtags = int(hashtag_metrics["node"].nunique()) if "node" in hashtag_metrics.columns and not hashtag_metrics.empty else "—"
best_spearman_text = "—" if pd.isna(best_spearman) else f"{best_spearman:.3f}"
best_overlap_text = "—" if pd.isna(best_overlap) else f"{int(best_overlap)}/12"

cards = f"""
<div style="display:grid; grid-template-columns:repeat(4,1fr); gap:14px; margin:16px 0;">
  <div style="background:#111827;color:white;padding:18px;border-radius:16px;box-shadow:0 8px 24px rgba(0,0,0,.12)">
    <div style="font-size:13px;color:#cbd5e1">Senatorial candidates</div>
    <div style="font-size:32px;font-weight:800">{n_candidates}</div>
    <div style="font-size:12px;color:#94a3b8">Ground-truth election outcome table</div>
  </div>
  <div style="background:#1e3a8a;color:white;padding:18px;border-radius:16px;box-shadow:0 8px 24px rgba(0,0,0,.12)">
    <div style="font-size:13px;color:#dbeafe">Best Spearman alignment</div>
    <div style="font-size:32px;font-weight:800">{best_spearman_text}</div>
    <div style="font-size:12px;color:#bfdbfe">Metric: {best_corr_metric}</div>
  </div>
  <div style="background:#7f1d1d;color:white;padding:18px;border-radius:16px;box-shadow:0 8px 24px rgba(0,0,0,.12)">
    <div style="font-size:13px;color:#fee2e2">Best Top-12 overlap</div>
    <div style="font-size:32px;font-weight:800">{best_overlap_text}</div>
    <div style="font-size:12px;color:#fecaca">Metric: {best_overlap_metric}</div>
  </div>
  <div style="background:#713f12;color:white;padding:18px;border-radius:16px;box-shadow:0 8px 24px rgba(0,0,0,.12)">
    <div style="font-size:13px;color:#fef3c7">Hashtag network nodes</div>
    <div style="font-size:32px;font-weight:800">{n_hashtags}</div>
    <div style="font-size:12px;color:#fde68a">Online discourse vocabulary network</div>
  </div>
</div>
"""
HTML(cards)

## 4. Actual Senate outcome: national vote ranking

This chart shows the ground-truth electoral outcome. The top 12 candidates are the elected Senate winners and serve as the benchmark for online-metric comparison.

In [ ]:
progress.step("4. Actual Senate outcome: national vote ranking")
if not national.empty and {"candidate", "total_votes", "winner_top12"}.issubset(national.columns):
    top_national = national.sort_values("total_votes", ascending=False).head(20).copy()
    top_national["winner_label"] = np.where(top_national["winner_top12"], "Top 12 winner", "Non-winner")
    fig = px.bar(
        top_national.sort_values("total_votes"),
        x="total_votes", y="candidate", color="winner_label", orientation="h",
        title="Actual Senate Vote Ranking — Top 20 Candidates",
        labels={"total_votes": "Total votes", "candidate": "Candidate", "winner_label": "Outcome"},
        text="vote_rank",
        color_discrete_sequence=["#2563eb", "#9ca3af"]
    )
    fig.update_traces(texttemplate="Rank %{text}", textposition="outside")
    fig.update_layout(height=720, template="plotly_white", legend_title_text="Outcome")
    fig.write_html(str(INTERACTIVE / "14_actual_vote_ranking_top20.html"))
    try:
        fig.write_image(str(FIGURES / "14_actual_vote_ranking_top20.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("National results table is missing the required columns.")

### Interpretation

The actual vote ranking is the reference point for all comparison. If a candidate ranks high online but low here, that indicates online overrepresentation. If a candidate ranks high here but lower online, that suggests offline strength, name recall, regional machinery, or voter bases less visible on Twitter/X.

## 5. Which online metrics aligned most strongly with votes?

This visualization ranks online metrics by Spearman correlation with actual vote totals. Spearman correlation is appropriate because we are mainly comparing **rank alignment**, not assuming a linear vote-prediction relationship.

In [ ]:
progress.step("5. Which online metrics aligned most strongly with votes?")
if not correlations.empty and {"online_metric", "spearman_r_vs_votes"}.issubset(correlations.columns):
    corr_plot = correlations.dropna(subset=["spearman_r_vs_votes"]).sort_values("spearman_r_vs_votes", ascending=True).copy()
    fig = px.bar(
        corr_plot,
        x="spearman_r_vs_votes", y="online_metric", orientation="h",
        title="Online Metrics vs Actual Votes — Spearman Rank Correlation",
        labels={"spearman_r_vs_votes": "Spearman correlation with total votes", "online_metric": "Online metric"},
        text="spearman_r_vs_votes",
        color="spearman_r_vs_votes",
        color_continuous_scale="Blues"
    )
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    fig.update_layout(height=650, template="plotly_white", coloraxis_showscale=False)
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="gray")
    fig.write_html(str(INTERACTIVE / "14_metric_correlations_with_votes.html"))
    try:
        fig.write_image(str(FIGURES / "14_metric_correlations_with_votes.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Correlation table is missing or incomplete.")

### Interpretation

The strongest online signals are expected to be **network-visibility metrics**, such as candidate mentions, co-mention weighted degree, unique hashtags, and PageRank. This is the strongest evidence that the project is not merely counting tweets; it is measuring how candidates are embedded in a political communication network.

## 6. Vote rank vs online mention rank

This scatterplot compares actual vote rank with online mention rank. Points near the diagonal show alignment. Points far from the diagonal show candidates who were overrepresented or underrepresented online.

In [ ]:
progress.step("6. Vote rank vs online mention rank")
if not comparison.empty and {"candidate", "vote_rank", "twitter_mention_count_rank", "winner_top12"}.issubset(comparison.columns):
    df = comparison.copy()
    df["winner_label"] = np.where(df["winner_top12"], "Top 12 winner", "Non-winner")
    fig = px.scatter(
        df, x="vote_rank", y="twitter_mention_count_rank", color="winner_label",
        hover_name="candidate",
        hover_data=["total_votes", "twitter_mention_count"],
        title="Actual Vote Rank vs Twitter/X Mention Rank",
        labels={"vote_rank": "Actual vote rank (1 = highest votes)", "twitter_mention_count_rank": "Twitter/X mention rank (1 = most mentioned)", "winner_label": "Outcome"},
        color_discrete_map={"Top 12 winner": "#2563eb", "Non-winner": "#ef4444"}
    )
    max_rank = int(max(df["vote_rank"].max(), df["twitter_mention_count_rank"].max()))
    fig.add_trace(go.Scatter(x=[1, max_rank], y=[1, max_rank], mode="lines", name="Perfect rank alignment", line=dict(dash="dash", color="gray")))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(autorange="reversed")
    fig.update_layout(height=720, template="plotly_white")
    fig.write_html(str(INTERACTIVE / "14_vote_rank_vs_mention_rank.html"))
    try:
        fig.write_image(str(FIGURES / "14_vote_rank_vs_mention_rank.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Comparison table is missing rank columns.")

### Interpretation

This chart is one of the clearest ways to explain the project. Candidates close to the diagonal have online visibility that roughly matches their actual electoral rank. Candidates above or below the diagonal reveal mismatches between Twitter/X discourse and actual voting behavior.

## 7. Online overperformance and underperformance

The overperformance index compares a candidate's actual vote rank with their online rank.

**Formula:** `Online overperformance = actual vote rank − online rank`

- Positive value: the candidate ranked better online than in actual votes.
- Near zero: online and vote ranking were aligned.
- Negative value: the candidate performed better electorally than online.

In [ ]:
progress.step("7. Online overperformance and underperformance")
if not comparison.empty and {"candidate", "twitter_mention_count_overperformance"}.issubset(comparison.columns):
    metric = "twitter_mention_count_overperformance"
    over = comparison[["candidate", "vote_rank", "twitter_mention_count_rank", metric, "winner_top12"]].dropna().copy()
    over = pd.concat([
        over.sort_values(metric, ascending=False).head(12),
        over.sort_values(metric, ascending=True).head(12)
    ]).drop_duplicates("candidate")
    over["direction"] = np.where(over[metric] >= 0, "Online overrepresented", "Electorally stronger than online")
    fig = px.bar(
        over.sort_values(metric),
        x=metric, y="candidate", color="direction", orientation="h",
        title="Online Overperformance / Underperformance Based on Twitter/X Mention Rank",
        labels={metric: "Actual vote rank − online mention rank", "candidate": "Candidate", "direction": "Interpretation"},
        hover_data=["vote_rank", "twitter_mention_count_rank", "winner_top12"],
        color_discrete_map={"Online overrepresented": "#dc2626", "Electorally stronger than online": "#2563eb"}
    )
    fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")
    fig.update_layout(height=780, template="plotly_white")
    fig.write_html(str(INTERACTIVE / "14_online_overperformance_underperformance.html"))
    try:
        fig.write_image(str(FIGURES / "14_online_overperformance_underperformance.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Comparison table is missing overperformance columns.")

### Interpretation

This figure is important because it makes the limitation visible. Twitter/X does not simply measure support. It also captures controversy, criticism, campaign activity, media attention, and urban/online political conversation.

## 8. Top-12 overlap by online metric

This chart shows how many of the actual Senate winners appear in the online top 12 for each metric.

In [ ]:
progress.step("8. Top-12 overlap by online metric")
if not top12_overlap.empty and {"online_metric", "top12_overlap_count"}.issubset(top12_overlap.columns):
    overlap = top12_overlap.sort_values("top12_overlap_count", ascending=True).copy()
    fig = px.bar(
        overlap,
        x="top12_overlap_count", y="online_metric", orientation="h",
        title="How Many Actual Senate Winners Were Captured by Each Online Top-12 Metric?",
        labels={"top12_overlap_count": "Actual winners found in online Top 12", "online_metric": "Online metric"},
        text="top12_overlap_count",
        color="top12_overlap_count",
        color_continuous_scale="Blues"
    )
    fig.update_traces(texttemplate="%{text}/12", textposition="outside")
    fig.update_xaxes(range=[0, 12])
    fig.update_layout(height=650, template="plotly_white", coloraxis_showscale=False)
    fig.write_html(str(INTERACTIVE / "14_top12_overlap_by_metric.html"))
    try:
        fig.write_image(str(FIGURES / "14_top12_overlap_by_metric.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Top-12 overlap table is missing or incomplete.")

### Interpretation

A high top-12 overlap suggests that the online metric captures electoral prominence. However, less-than-perfect overlap means Twitter/X cannot be treated as a direct election predictor.

## 9. Winner vs non-winner comparison across online metrics

This visualization compares the distributions of key online indicators between actual Top 12 winners and non-winners.

In [ ]:
progress.step("9. Winner vs non-winner comparison across online metrics")
key_metrics = [
    "twitter_mention_count", "total_engagement", "viewCount",
    "comention_weighted_degree", "comention_pagerank",
    "unique_hashtags", "total_candidate_hashtag_weight", "bipartite_pagerank"
]
if not comparison.empty and "winner_top12" in comparison.columns:
    available = [m for m in key_metrics if m in comparison.columns]
    long = comparison[["candidate", "winner_top12"] + available].melt(
        id_vars=["candidate", "winner_top12"], var_name="online_metric", value_name="value"
    ).dropna()
    long["outcome"] = np.where(long["winner_top12"], "Top 12 winner", "Non-winner")
    long["log_value"] = np.log1p(long["value"])
    fig = px.box(
        long, x="online_metric", y="log_value", color="outcome", points="all",
        title="Winner vs Non-Winner Distributions Across Online Metrics",
        labels={"online_metric": "Online metric", "log_value": "log(1 + metric value)", "outcome": "Election outcome"},
        color_discrete_map={"Top 12 winner": "#2563eb", "Non-winner": "#ef4444"}
    )
    fig.update_layout(height=750, template="plotly_white", xaxis_tickangle=-35)
    fig.write_html(str(INTERACTIVE / "14_winner_nonwinner_metric_distributions.html"))
    try:
        fig.write_image(str(FIGURES / "14_winner_nonwinner_metric_distributions.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Comparison table is missing winner labels.")

### Interpretation

If winner distributions sit higher than non-winner distributions, this suggests that actual winners were more visible or central online. Overlap between the boxes reminds us that some non-winners were also highly visible online.

## 10. Hashtag network: central hashtags and bridge hashtags

A hashtag can be important in two different ways:

1. **Centrality / PageRank:** the hashtag is structurally influential in the network.
2. **Betweenness:** the hashtag bridges different communities.

This distinction is essential to the network science interpretation.

In [ ]:
progress.step("10. Hashtag network: central hashtags and bridge hashtags")
if not hashtag_metrics.empty and {"node", "pagerank", "betweenness_centrality", "weighted_degree"}.issubset(hashtag_metrics.columns):
    top_pr = hashtag_metrics.sort_values("pagerank", ascending=False).head(15).copy()
    fig = px.bar(
        top_pr.sort_values("pagerank"),
        x="pagerank", y="node", orientation="h",
        title="Most Structurally Central Hashtags by PageRank",
        labels={"pagerank": "PageRank", "node": "Hashtag"},
        text="pagerank",
        color="pagerank",
        color_continuous_scale="Blues"
    )
    fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
    fig.update_layout(height=620, template="plotly_white", coloraxis_showscale=False)
    fig.write_html(str(INTERACTIVE / "14_top_hashtags_pagerank.html"))
    try:
        fig.write_image(str(FIGURES / "14_top_hashtags_pagerank.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()

    top_bridge = hashtag_metrics.sort_values("betweenness_centrality", ascending=False).head(15).copy()
    fig = px.bar(
        top_bridge.sort_values("betweenness_centrality"),
        x="betweenness_centrality", y="node", orientation="h",
        title="Top Bridge Hashtags by Betweenness Centrality",
        labels={"betweenness_centrality": "Betweenness centrality", "node": "Hashtag"},
        text="betweenness_centrality",
        color="betweenness_centrality",
        color_continuous_scale="Oranges"
    )
    fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
    fig.update_layout(height=620, template="plotly_white", coloraxis_showscale=False)
    fig.write_html(str(INTERACTIVE / "14_top_bridge_hashtags.html"))
    try:
        fig.write_image(str(FIGURES / "14_top_bridge_hashtags.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Hashtag metrics table is missing required centrality columns.")

### Interpretation

The most central hashtags act as common election-discourse hubs. Bridge hashtags are especially important because they connect otherwise separate communities, issues, candidates, or media conversations.

## 11. Popularity vs structural importance among hashtags

A hashtag can be frequently used but not structurally important, or structurally important even if it is not the most frequent. This scatterplot helps separate popularity from network position.

In [ ]:
progress.step("11. Popularity vs structural importance among hashtags")
if not hashtag_metrics.empty and {"node", "weighted_degree", "pagerank", "betweenness_centrality"}.issubset(hashtag_metrics.columns):
    h = hashtag_metrics.copy()
    h["log_weighted_degree"] = np.log1p(h["weighted_degree"])
    h["label"] = np.where(h["pagerank"].rank(ascending=False) <= 10, h["node"], "")
    fig = px.scatter(
        h, x="log_weighted_degree", y="pagerank", size="betweenness_centrality",
        hover_name="node", text="label",
        title="Hashtag Popularity vs Network Importance",
        labels={"log_weighted_degree": "log(1 + weighted degree)", "pagerank": "PageRank", "betweenness_centrality": "Betweenness"},
        opacity=0.75
    )
    fig.update_traces(textposition="top center")
    fig.update_layout(height=720, template="plotly_white")
    fig.write_html(str(INTERACTIVE / "14_hashtag_popularity_vs_importance.html"))
    try:
        fig.write_image(str(FIGURES / "14_hashtag_popularity_vs_importance.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Hashtag metrics table is missing required columns.")

## 12. Candidate co-mention network graph

This graph visualizes which candidates were discussed together. Edge thickness is based on co-mention frequency. Node size is based on PageRank or weighted degree. This graph represents **discourse association**, not endorsement.

In [ ]:
progress.step("12. Candidate co-mention network graph")
def draw_candidate_network(edges, metrics, max_edges=60, min_weight=None):
    if edges.empty:
        print("Candidate co-mention edge table is missing.")
        return
    e = edges.copy()
    # Standardize possible column names.
    if "source" not in e.columns and "candidate_a" in e.columns:
        e = e.rename(columns={"candidate_a": "source", "candidate_b": "target"})
    if "source" not in e.columns:
        e = e.rename(columns={e.columns[0]: "source", e.columns[1]: "target"})
    if min_weight is not None:
        e = e[e["weight"] >= min_weight]
    e = e.sort_values("weight", ascending=False).head(max_edges)
    G = graph_from_edges(e, source="source", target="target", weight="weight", directed=False)
    if G.number_of_nodes() == 0:
        print("No candidate graph to draw after filtering.")
        return
    metric_map = {}
    if not metrics.empty and "node" in metrics.columns:
        metric_map = metrics.set_index("node").to_dict(orient="index")
    communities = detect_communities(G)
    pos = nx.spring_layout(G, seed=42, weight="weight", k=0.75)
    fig, ax = plt.subplots(figsize=(16, 11))
    weights = [G[u][v].get("weight", 1) for u, v in G.edges()]
    max_w = max(weights) if weights else 1
    widths = [0.7 + 5 * (w / max_w) for w in weights]
    node_sizes = []
    colors = []
    for n in G.nodes():
        m = metric_map.get(n, {})
        size_base = m.get("pagerank", None)
        if size_base is None or pd.isna(size_base):
            size_base = m.get("weighted_degree", G.degree(n, weight="weight"))
            node_sizes.append(280 + 22 * np.sqrt(size_base))
        else:
            node_sizes.append(400 + 14000 * float(size_base))
        colors.append(communities.get(n, 0))
    nx.draw_networkx_edges(G, pos, ax=ax, width=widths, alpha=0.28, edge_color="#374151")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color=colors, cmap="tab20", alpha=0.92, edgecolors="white", linewidths=1.3)
    # Label only central nodes and endpoints of strongest edges to reduce clutter.
    important_nodes = set()
    if not metrics.empty and "node" in metrics.columns:
        top_nodes = metrics.sort_values("pagerank" if "pagerank" in metrics.columns else "weighted_degree", ascending=False).head(20)["node"].tolist()
        important_nodes.update(top_nodes)
    important_nodes.update(e.head(15)["source"].tolist())
    important_nodes.update(e.head(15)["target"].tolist())
    labels = {n: n for n in G.nodes() if n in important_nodes}
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=9, font_weight="bold")
    ax.set_title("Candidate Co-Mention Network — Core Discourse Associations", fontsize=18, fontweight="bold", pad=18)
    ax.text(0.01, -0.04, "Node size: PageRank / weighted degree | Edge width: co-mention frequency | Color: detected network community", transform=ax.transAxes, fontsize=11, color="#4b5563")
    ax.axis("off")
    out = FIGURES / "14_candidate_comention_network_core.png"
    fig.tight_layout()
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

draw_candidate_network(candidate_edges, candidate_metrics, max_edges=75)

### Interpretation

The candidate co-mention network is central to the network science story. Candidates who are frequently co-mentioned with many others occupy structurally prominent positions in the political discourse. However, co-mention must be interpreted carefully: it can represent support, comparison, criticism, controversy, or bloc association.

## 13. Strongest candidate co-mention pairs

This chart shows the strongest candidate-pair associations in the discourse.

In [ ]:
progress.step("13. Strongest candidate co-mention pairs")
if not candidate_edges.empty:
    e = candidate_edges.copy()
    if "source" not in e.columns:
        if {"candidate_a", "candidate_b"}.issubset(e.columns):
            e = e.rename(columns={"candidate_a": "source", "candidate_b": "target"})
        else:
            e = e.rename(columns={e.columns[0]: "source", e.columns[1]: "target"})
    e["pair"] = e["source"].astype(str) + " — " + e["target"].astype(str)
    top_pairs = e.sort_values("weight", ascending=False).head(20).sort_values("weight")
    fig = px.bar(
        top_pairs, x="weight", y="pair", orientation="h",
        title="Strongest Candidate Co-Mention Pairs",
        labels={"weight": "Number of co-mentions", "pair": "Candidate pair"},
        text="weight", color="weight", color_continuous_scale="Blues"
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(height=750, template="plotly_white", coloraxis_showscale=False)
    fig.write_html(str(INTERACTIVE / "14_strongest_candidate_comention_pairs.html"))
    try:
        fig.write_image(str(FIGURES / "14_strongest_candidate_comention_pairs.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Candidate co-mention edges are missing.")

## 14. Candidate–hashtag diversity

This chart shows which candidates were associated with the broadest range of hashtags. This is a useful measure of online narrative spread.

In [ ]:
progress.step("14. Candidate–hashtag diversity")
if not candidate_hashtag_diversity.empty and {"candidate", "unique_hashtags", "total_candidate_hashtag_weight"}.issubset(candidate_hashtag_diversity.columns):
    div = candidate_hashtag_diversity.sort_values("unique_hashtags", ascending=False).head(20).sort_values("unique_hashtags")
    fig = px.bar(
        div, x="unique_hashtags", y="candidate", orientation="h",
        title="Candidates With the Broadest Hashtag Diversity",
        labels={"unique_hashtags": "Unique hashtags linked to candidate", "candidate": "Candidate"},
        text="unique_hashtags", color="total_candidate_hashtag_weight", color_continuous_scale="YlOrRd"
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(height=720, template="plotly_white", coloraxis_colorbar_title="Total hashtag weight")
    fig.write_html(str(INTERACTIVE / "14_candidate_hashtag_diversity.html"))
    try:
        fig.write_image(str(FIGURES / "14_candidate_hashtag_diversity.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Candidate-hashtag diversity table is missing or incomplete.")

### Interpretation

Hashtag diversity helps distinguish candidates who were tied to a narrow set of campaign tags from candidates who appeared across many narratives, issues, communities, or controversies.

## 15. Candidate-hashtag network: top candidate and hashtag nodes

This chart combines candidate and hashtag nodes from the bipartite network. It helps identify whether the network is dominated by election-wide hashtags, candidate names, issue tags, or public discourse labels.

In [ ]:
progress.step("15. Candidate-hashtag network: top candidate and hashtag nodes")
if not candidate_hashtag_metrics.empty and {"node", "pagerank", "node_type"}.issubset(candidate_hashtag_metrics.columns):
    top_bi = candidate_hashtag_metrics.sort_values("pagerank", ascending=False).head(25).copy()
    fig = px.bar(
        top_bi.sort_values("pagerank"), x="pagerank", y="node", color="node_type", orientation="h",
        title="Most Central Nodes in the Candidate–Hashtag Bipartite Network",
        labels={"pagerank": "PageRank", "node": "Node", "node_type": "Node type"},
        text="pagerank",
        color_discrete_map={"candidate": "#2563eb", "hashtag": "#f59e0b"}
    )
    fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
    fig.update_layout(height=750, template="plotly_white")
    fig.write_html(str(INTERACTIVE / "14_candidate_hashtag_top_nodes.html"))
    try:
        fig.write_image(str(FIGURES / "14_candidate_hashtag_top_nodes.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Candidate-hashtag metrics are missing node type or PageRank columns.")

## 16. Geographic comparison: NCR vs national results

Twitter/X discourse may be more similar to urban political attention than to national voting behavior. This section now shows the **complete NCR-vs-national rank table**, a rank-alignment scatterplot, and a rank-difference chart. If the upstream NCR table is missing, the notebook rebuilds it directly from the raw Senate results dataset.

In [ ]:
progress.step("16. Geographic comparison: NCR vs national results")

if validate_ncr_vs_national_table(ncr_vs_national):
    ncr = ncr_vs_national.copy().sort_values("regional_rank")
    ncr["winner_label"] = np.where(ncr["winner_top12"], "Top 12 winner", "Non-winner")
    ncr["regional_vote_share_pct"] = ncr["regional_vote_share"] * 100

    display_cols = [
        "regional_rank", "candidate", "party", "votes", "regional_vote_share_pct",
        "vote_rank", "total_votes", "winner_top12", "ncr_minus_national_rank"
    ]
    ncr_table = ncr[[c for c in display_cols if c in ncr.columns]].copy()
    ncr_table = ncr_table.rename(columns={
        "regional_rank": "NCR rank",
        "candidate": "Candidate",
        "party": "Party",
        "votes": "NCR votes",
        "regional_vote_share_pct": "NCR vote share (%)",
        "vote_rank": "National rank",
        "total_votes": "National votes",
        "winner_top12": "Top 12 winner",
        "ncr_minus_national_rank": "NCR rank - National rank"
    })

    # Save complete table for final report use.
    ncr_table.to_csv(TABLES / "14_ncr_vs_national_full_table.csv", index=False)
    try:
        ncr_table.to_html(TABLES / "14_ncr_vs_national_full_table.html", index=False)
    except Exception as e:
        print("HTML table export skipped:", e)

    display(Markdown("### Complete NCR vs National Rank Table"))
    display(Markdown("Negative rank difference means the candidate ranked higher in NCR than nationally. Positive rank difference means the candidate ranked lower in NCR than nationally."))
    display(ncr_table.style.format({
        "NCR votes": "{:,.0f}",
        "National votes": "{:,.0f}",
        "NCR vote share (%)": "{:.2f}",
        "NCR rank - National rank": "{:+.0f}"
    }).background_gradient(subset=["NCR rank - National rank"], cmap="RdBu_r"))

    fig = px.scatter(
        ncr, x="vote_rank", y="regional_rank", color="winner_label",
        size="votes", size_max=26,
        hover_name="candidate",
        hover_data={
            "vote_rank": True,
            "regional_rank": True,
            "total_votes": ":,",
            "votes": ":,",
            "regional_vote_share_pct": ":.2f",
            "ncr_minus_national_rank": ":+.0f",
            "winner_label": True
        },
        title="NCR Rank vs National Vote Rank",
        labels={
            "vote_rank": "National vote rank",
            "regional_rank": "NCR rank",
            "winner_label": "Outcome",
            "votes": "NCR votes"
        },
        color_discrete_map={"Top 12 winner": "#2563eb", "Non-winner": "#ef4444"}
    )
    max_rank = int(max(ncr["vote_rank"].max(), ncr["regional_rank"].max()))
    fig.add_trace(go.Scatter(
        x=[1, max_rank], y=[1, max_rank], mode="lines", name="Perfect rank alignment",
        line=dict(dash="dash", color="gray")
    ))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(autorange="reversed")
    fig.update_layout(height=720, template="plotly_white")
    fig.write_html(str(INTERACTIVE / "14_ncr_vs_national_rank_alignment.html"))
    try:
        fig.write_image(str(FIGURES / "14_ncr_vs_national_rank_alignment.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()

    # Rank-shift chart: strongest NCR-overrepresented and NCR-underrepresented candidates.
    rank_shift = ncr.sort_values("ncr_minus_national_rank")
    rank_shift_focus = pd.concat([rank_shift.head(10), rank_shift.tail(10)]).drop_duplicates()
    fig_shift = px.bar(
        rank_shift_focus.sort_values("ncr_minus_national_rank"),
        x="ncr_minus_national_rank", y="candidate", orientation="h",
        color="winner_label",
        title="Largest NCR-vs-National Rank Differences",
        labels={
            "ncr_minus_national_rank": "NCR rank - National rank",
            "candidate": "Candidate",
            "winner_label": "Outcome"
        },
        color_discrete_map={"Top 12 winner": "#2563eb", "Non-winner": "#ef4444"}
    )
    fig_shift.add_vline(x=0, line_dash="dash", line_color="gray")
    fig_shift.update_layout(height=760, template="plotly_white")
    fig_shift.write_html(str(INTERACTIVE / "14_ncr_vs_national_rank_difference.html"))
    try:
        fig_shift.write_image(str(FIGURES / "14_ncr_vs_national_rank_difference.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig_shift.show()
else:
    print("NCR vs national rank table is still missing or incomplete. Check that data/raw/senate25-final_updated.csv exists and contains a region column.")


### Interpretation

This graph helps explain why Twitter/X may overrepresent certain candidates. If a candidate is stronger in NCR than nationally and also strong online, the online network may be reflecting urban political attention more than national voter behavior.

## 17. Regional turnout and ballot behavior context

This chart adds election-context analytics beyond network science. Turnout and undervote rates help describe the electoral environment in which online discourse is being compared.

In [ ]:
progress.step("17. Regional turnout and ballot behavior context")
if not regional_turnout.empty and {"region", "turnout_rate", "undervote_rate"}.issubset(regional_turnout.columns):
    rt = regional_turnout.copy().sort_values("turnout_rate")
    rt["turnout_pct"] = rt["turnout_rate"] * 100
    rt["undervote_pct"] = rt["undervote_rate"] * 100
    fig = go.Figure()
    fig.add_trace(go.Bar(y=rt["region"], x=rt["turnout_pct"], orientation="h", name="Turnout rate (%)"))
    fig.add_trace(go.Bar(y=rt["region"], x=rt["undervote_pct"], orientation="h", name="Undervote rate (%)"))
    fig.update_layout(
        title="Regional Turnout and Undervote Rates",
        xaxis_title="Percent", yaxis_title="Region",
        barmode="group", height=760, template="plotly_white"
    )
    fig.write_html(str(INTERACTIVE / "14_regional_turnout_undervote.html"))
    try:
        fig.write_image(str(FIGURES / "14_regional_turnout_undervote.png"), scale=2)
    except Exception as e:
        print("PNG export skipped:", e)
    fig.show()
else:
    print("Regional turnout table is missing or incomplete.")

## 18. Final research answer

This cell generates a concise, report-ready interpretation from the results above.

In [ ]:
progress.step("18. Final research answer")
# Build report-ready narrative using actual computed outputs.
summary_lines = []
summary_lines.append("# Final Interpretive Summary")
summary_lines.append("")
summary_lines.append("## Main research answer")
summary_lines.append("")
summary_lines.append("The Philippine Election 2025 Twitter/X discourse network partially reflected actual senatorial election outcomes. Candidates who were more visible, more connected, and more embedded in hashtag and co-mention networks generally received more votes. However, Twitter/X did not perfectly reproduce the final Top 12 Senate results, indicating that the platform captures online attention and discourse structure rather than voter behavior directly.")
summary_lines.append("")

if not correlations.empty and "spearman_r_vs_votes" in correlations.columns:
    top_corr = correlations.sort_values("spearman_r_vs_votes", ascending=False).head(5)
    summary_lines.append("## Strongest online-outcome alignments")
    summary_lines.append("")
    for _, r in top_corr.iterrows():
        summary_lines.append(f"- **{r['online_metric']}**: Spearman r = **{r['spearman_r_vs_votes']:.3f}**")
    summary_lines.append("")

if not top12_overlap.empty and "top12_overlap_count" in top12_overlap.columns:
    best = top12_overlap.sort_values("top12_overlap_count", ascending=False).iloc[0]
    summary_lines.append("## Top-12 overlap")
    summary_lines.append("")
    summary_lines.append(f"The best online metric, **{best['online_metric']}**, captured **{int(best['top12_overlap_count'])} out of 12** actual Senate winners in its online Top 12.")
    summary_lines.append("")

summary_lines.append("## Network science interpretation")
summary_lines.append("")
summary_lines.append("The strongest findings come from network-based measures rather than raw virality alone. Candidate mention count, co-mention weighted degree, PageRank, unique hashtag diversity, and candidate-hashtag association strength show how deeply a candidate is embedded in the online election discourse network. These metrics are more meaningful than simple likes or retweets because they describe structural position, not just popularity.")
summary_lines.append("")
summary_lines.append("## Limitation")
summary_lines.append("")
summary_lines.append("Twitter/X should not be treated as a representative sample of Filipino voters. It is likely biased toward online, urban, politically expressive, and media-engaged users. Therefore, mismatches between Twitter/X rank and vote rank are not errors; they are evidence that online discourse and electoral behavior are related but distinct social systems.")
summary_lines.append("")
summary_lines.append("## Final conclusion")
summary_lines.append("")
summary_lines.append("Twitter/X network analysis is useful for understanding political attention, discourse communities, bridge hashtags, candidate associations, and online amplification. It can partially align with electoral outcomes, but its greatest value is explanatory rather than predictive: it reveals how election discourse is structured online and how that structure relates to, but does not fully determine, actual voting results.")

summary_md = "\n".join(summary_lines)
display(Markdown(summary_md))

out_md = REPORT_ASSETS / "14_final_interpretive_summary.md"
out_md.write_text(summary_md, encoding="utf-8")
print("Saved narrative summary:", out_md)

## 19. Export a visual index

This cell creates an index of the final figures and interactive files produced by this notebook.

In [ ]:
progress.step("19. Export a visual index")
visual_files = []
for folder, label in [(FIGURES, "Static figure"), (INTERACTIVE, "Interactive HTML")]:
    for p in sorted(folder.glob("14_*")):
        visual_files.append({"type": label, "file": p.name, "path": str(p.relative_to(ROOT))})
visual_index = pd.DataFrame(visual_files)
if not visual_index.empty:
    display(visual_index)
    out = TABLES / "14_final_interpretive_visual_index.csv"
    visual_index.to_csv(out, index=False)
    print("Saved visual index:", out)
else:
    print("No visual files were generated yet. Run the visualization cells above.")

# Final note

This notebook is the discussion layer of the project. The earlier notebooks are responsible for computing the results. This notebook turns those results into a visually supported argument:

> **Twitter/X election discourse partially reflects electoral outcomes, but its deeper value is in revealing networked political attention, discourse clustering, bridge topics, and online-overrepresentation patterns.**

## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
